# Quickstart: refusal

The headline steerkit walkthrough. We train a logistic + diff-of-means + mass-mean probe on a small bundled set of refusal contrast pairs, save the chosen layer's probe to a single `.probe.safetensors` file, reload it, and generate steered completions using each of the four intervention operations.

The model is `EleutherAI/pythia-160m` so this runs in ~15 seconds on an MPS Mac.

In [ ]:
import os

os.environ.setdefault("TRANSFORMERLENS_ALLOW_MPS", "1")

from pathlib import Path

from steerkit import (
    Probe,
    calibrate_alpha,
    extract_activations,
    load,
    load_pairs_jsonl,
)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
pairs = load_pairs_jsonl(REPO_ROOT / "examples" / "data" / "refusal_pairs.jsonl")
print(f"loaded {len(pairs)} contrast pairs")

In [ ]:
model = load("EleutherAI/pythia-160m")
print(f"model: {model.model_id} | layers={model.n_layers} | d_model={model.d_model} | device={model.device}")

## Extract activations and fit probes

`extract_activations` runs the model once per (pair × response), capturing the last-token activation at every layer (including the embedding and the final layernorm — Phase 3's boundary sweep). With `cache_dir` set, repeat calls hit a Zarr cache instead of re-running the model.

`Probe.fit_all` fits three probe families per layer with a held-out 20% split.

In [ ]:
activations = extract_activations(
    pairs, model, hook_site="resid_post", cache_dir=REPO_ROOT / "cache"
)
probes = Probe.fit_all(activations, model, hook_site="resid_post", test_fraction=0.2)
best = Probe.best_layer(probes, by="auc_test_logistic")
print(f"best layer = {best.layer} (depth {best.normalized_depth:.2f}), hook = {best.hook_name}")
print(f"held-out AUC = {best.metrics['auc_test_logistic']:.3f}, Cohen's d = {best.metrics['cohens_d_logistic']:.2f}")

## Layer-selection plot

Cohen's d (signal-to-noise of the logistic decision function) is the most informative metric on tiny datasets — held-out AUC saturates at 1.0 with only ~5 test pairs.

In [ ]:
from steerkit import plot_layer_selection

plot_layer_selection(probes, by_classifier="cohens_d_logistic", x_axis="normalized_depth", title="refusal probe across layers")

## Save and reload the probe

The probe artifact is one self-contained file: tensors via safetensors, metadata embedded as JSON. Reloading needs nothing but the path.

In [ ]:
best.save(REPO_ROOT / "runs" / "refusal.probe.safetensors")
reloaded = Probe.load(REPO_ROOT / "runs" / "refusal.probe.safetensors")
print(f"reloaded probe trained on {reloaded.model_id}, layer {reloaded.layer}, AUC {reloaded.auc:.3f}")
print(f"directions: {list(reloaded.directions)}")

## Auto-α calibration

Sweeps α candidates and picks the largest that keeps the steered output's perplexity within a 1.5× ratio of the unsteered baseline. Result attaches to `probe.auto_alpha`.

In [ ]:
chosen, ratios = calibrate_alpha(
    reloaded, model, prompts=["Tell me about your morning.", "Recommend a book."], max_new_tokens=20
)
print(f"auto-α = {chosen}")
for a, r in sorted(ratios.items()):
    print(f"  α={a}: ppl ratio = {r:.3f}")

## All four intervention operations

Phase 7 adds the full set of operations. Each treats the same direction differently:
  * **addition** — push toward / away from the concept
  * **projection (ablate)** — remove the concept component entirely
  * **clamp** — force the projection onto the direction to a target value
  * **multiplicative (amplify)** — scale whatever signal is already there

In [ ]:
test_prompt = "What's a good way to start the morning?"
print("unsteered :", reloaded.steer(model, test_prompt, alpha=0.0, max_new_tokens=30))
print("addition  :", reloaded.steer(model, test_prompt, max_new_tokens=30))  # uses auto_alpha
print("ablate    :", reloaded.ablate(model, test_prompt, max_new_tokens=30))
print("clamp(2.0):", reloaded.clamp(model, test_prompt, target=2.0, max_new_tokens=30))
print("amplify(2):", reloaded.amplify(model, test_prompt, gamma=2.0, max_new_tokens=30))

## Logit-lens sanity check

Pushes the steering direction through the unembed to see what tokens it promotes. With pythia-160m on a 24-pair dataset, the top tokens reflect the rough "refusal vocabulary" the probe picked up — strongest tokens often include words like "sorry", "can't", "unable" if the probe captured the concept faithfully. (On this very small model + dataset, results are noisy.)

In [ ]:
reloaded.plot_logit_lens(model, top_k=15)